# Día 3 · Calibración del juez de relevancia v3

Este notebook valida primero el juez contra 30 etiquetas humanas. Solo si alcanza Accuracy y F1 ≥ 0,70 reevalúa los 178 pares normal/HyDE.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-03-experimento-hyde
!pip -q install pandas openpyxl openai scikit-learn

## 1. Subir dos archivos
Selecciona simultáneamente `resultados_hyde_dia_03.zip` y `human_audit_sample_blind(2).xlsx`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, pandas as pd
uploaded = files.upload()
zip_name = next(n for n in uploaded if n.lower().endswith('.zip'))
audit_name = next(n for n in uploaded if n.lower().endswith('.xlsx'))
input_dir = Path('/content/hyde_v1'); input_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive: archive.extractall(input_dir)
human = pd.read_excel(audit_name)
key = pd.read_excel(input_dir / 'human_audit_sample_key.xlsx')
audit = key.merge(human[['question_id','chunk_id','control_humano_relevante']], on=['question_id','chunk_id'], validate='one_to_one')
audit.to_csv('/content/audit_candidates.csv', index=False, encoding='utf-8-sig')
print('Etiquetas humanas:', human.control_humano_relevante.value_counts().to_dict())

## 2. Cargar la clave temporal

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

## 3. Validar el juez con la muestra humana

In [ ]:
!python -m src.retrieval.rejudge_hyde_results --results /content/audit_candidates.csv --questions data/evaluation/gold_questions.csv --output-dir /content/calibracion_juez_v3

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix
calibration_file = Path('/content/calibracion_juez_v3/retrieval_results_rejudged_v3.csv')
if not calibration_file.exists():
    raise RuntimeError('La calibración no terminó. Revisa el error de la celda anterior; este archivo es una salida, no un archivo que debas subir.')
cal = pd.read_csv(calibration_file)
y = cal.control_humano_relevante.astype(int); p = cal.relevant.astype(int)
tn, fp, fn, tp = confusion_matrix(y,p,labels=[0,1]).ravel()
metrics = {'TP':int(tp),'TN':int(tn),'FP':int(fp),'FN':int(fn),'accuracy':accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'f1':f1_score(y,p,zero_division=0),'kappa':cohen_kappa_score(y,p)}
display(pd.DataFrame([metrics]))
gate = metrics['accuracy'] >= .70 and metrics['f1'] >= .70
print('COMPUERTA:', 'APROBADA' if gate else 'NO APROBADA')

## 4. Reevaluar el experimento únicamente si se aprobó la compuerta

In [ ]:
import subprocess, shutil
if gate:
    subprocess.run(['python','-m','src.retrieval.rejudge_hyde_results','--results',str(input_dir/'retrieval_results_evaluated.csv'),'--questions','data/evaluation/gold_questions.csv','--output-dir','/content/resultados_hyde_juez_v3'], check=True)
    display(pd.read_csv('/content/resultados_hyde_juez_v3/metrics_by_method_v3.csv'))
    out = shutil.make_archive('/content/resultados_hyde_juez_v3','zip','/content/resultados_hyde_juez_v3')
else:
    out = shutil.make_archive('/content/calibracion_juez_v3','zip','/content/calibracion_juez_v3')
files.download(out)